In [2]:
# =====================================================
# PHASE 2 - Multi-Field Automatic + Fuzzy Matching
# According to Professor Lin's instructions
# =====================================================

import pandas as pd
from thefuzz import fuzz
from pathlib import Path

pd.set_option('display.max_colwidth', 100)

# ------------------- LOAD DATA -------------------
mh_search = pd.read_csv('phase1_output/mh_search_cleaned.csv')
at_search = pd.read_csv('phase1_output/at_search_cleaned.csv')

mh_gt = pd.read_csv('phase1_output/mh_gt_cleaned.csv')
at_gt = pd.read_csv('phase1_output/at_gt_cleaned.csv')

# ------------------- MAIN MATCHING FUNCTION -------------------
def advanced_matching(search_df, gt_df, domain="MH", fuzzy_threshold=75):
    results = []
    
    # Group GT by city
    gt_by_city = {}
    for _, gt_row in gt_df.iterrows():
        city = str(gt_row['City']).lower().strip()
        gt_by_city.setdefault(city, []).append(gt_row)
    
    print(f"\n🔍 Processing {domain} with multi-field matching...")
    
    for _, s_row in search_df.iterrows():
        city = str(s_row['City']).lower().strip()
        if city not in gt_by_city:
            results.append({'is_match': False, **s_row.to_dict()})
            continue
            
        search_name = str(s_row['Clean_Service_Name'])
        search_phone = str(s_row.get('Phone number', ''))
        search_addr = str(s_row.get('address', ''))
        
        best_score = 0
        best_gt = None
        match_fields = 0
        
        for gt_row in gt_by_city[city]:
            gt_name = str(gt_row['Clean_Service_Name'])
            gt_phone = str(gt_row.get('Phone', ''))
            gt_addr = str(gt_row.get('Address', ''))
            
            # Count exact matches on multiple fields
            current_match_fields = 0
            if search_phone and gt_phone and search_phone in gt_phone:
                current_match_fields += 1
            if search_addr and gt_addr and search_addr.lower() in gt_addr.lower():
                current_match_fields += 1
            
            # Name similarity
            name_score = max(
                fuzz.token_sort_ratio(search_name, gt_name),
                fuzz.token_set_ratio(search_name, gt_name),
                fuzz.partial_ratio(search_name, gt_name)
            )
            
            if current_match_fields >= 2:
                # Automatic Match (2+ fields)
                is_match = True
                final_score = 100
            else:
                # Fuzzy Match (only 1 or 0 fields)
                final_score = name_score
                is_match = (final_score >= fuzzy_threshold)
            
            if final_score > best_score:
                best_score = final_score
                best_gt = gt_row['Name']
                match_fields = current_match_fields
        
        results.append({
            'id': s_row.get('id'),
            'Service name': s_row['Service name'],
            'Clean_Service_Name': search_name,
            'City': s_row['City'],
            'best_matched_gt': best_gt,
            'similarity_score': best_score,
            'matching_fields': match_fields,
            'is_match': is_match
        })
    
    return pd.DataFrame(results)

# ------------------- RUN FOR BOTH DOMAINS -------------------
mh_result = advanced_matching(mh_search, mh_gt, "Mental Health", fuzzy_threshold=72)
at_result = advanced_matching(at_search, at_gt, "Addiction Treatment", fuzzy_threshold=72)

# ------------------- SAVE -------------------
output_dir = Path('phase2_output')
output_dir.mkdir(exist_ok=True)

mh_result.to_csv(output_dir / 'mh_search_with_is_match.csv', index=False)
at_result.to_csv(output_dir / 'at_search_with_is_match.csv', index=False)

print("✅ Phase 2 Completed!")
print(f"MH Matches: {mh_result['is_match'].sum()}")
print(f"AT Matches: {at_result['is_match'].sum()}")


🔍 Processing Mental Health with multi-field matching...

🔍 Processing Addiction Treatment with multi-field matching...
✅ Phase 2 Completed!
MH Matches: 39
AT Matches: 99
